# Neutron-star EoS experiments

This notebook is a guided interface to the toolkit's existing analytical, CSV, and CompOSE workflows. It keeps source data, evaluated continuous barotropes, diagnostics, and stellar results visibly distinct.

> **Scientific boundary:** stellar integrations stop at the lowest supplied positive pressure. Every plotted radius is therefore a **source-boundary radius**, not a vacuum-surface radius. This notebook does not calculate tidal observables, infer stable branches or maximum masses, repair discontinuities, splice crusts, smooth inputs, or extrapolate beyond a declared domain.

## Setup and use

From the repository root, install the notebook dependencies with `python -m pip install -e ".[notebook]"`, start JupyterLab, edit the single parameter cell below, and use **Restart Kernel and Run All Cells**. The default run uses the bundled CSV and writes nothing.

In [ ]:
from __future__ import annotations

import hashlib
import json
import runpy
import shutil
from dataclasses import asdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import JSON, Markdown, display

from neutron_star_eos import (
    EosInputError,
    EosModel,
    StellarConfig,
    StellarSolveError,
    open_eos,
)
from neutron_star_eos.plotting import (
    plot_compose_closure_residuals,
    plot_compose_cold_residuals,
    plot_composition,
    plot_mass_profile,
    plot_mass_radius,
    plot_phase_codes,
    plot_pressure_energy,
    plot_sequence_status,
    plot_sound_speed_squared,
)

## Experiment controls

Choose one input kind. For an analytical model, edit `notebooks/analytical_eos.py`; the function definitions there are authoritative. CSV and CompOSE paths may be absolute or relative to the repository root. Beginner controls are the input choice, central pressure, sequence range, and point count. Solver tolerances, radius limits, CompOSE ordering policies, and diagnostic validation are expert controls whose exact values are recorded in saved output.

In [ ]:
# Repository and selected input
REPOSITORY_ROOT = (
    None  # Set explicitly only if the kernel starts outside this repository.
)
INPUT_KIND = "csv"  # "csv", "analytical", or "compose"

# Ordinary CSV controls
CSV_PATH = "examples/tabulated.csv"
CSV_NAME = None
CSV_SOURCE_DESCRIPTION = None
CSV_EPSILON_COLUMN = "epsilon_mev_fm3"
CSV_PRESSURE_COLUMN = "pressure_mev_fm3"
CSV_BARYON_DENSITY_COLUMN = None

# User-authored analytical function file
ANALYTICAL_DEFINITION_PATH = "notebooks/analytical_eos.py"

# CompOSE controls (required only when INPUT_KIND == "compose")
COMPOSE_PATH = None
COMPOSE_MODEL_ID = None
COMPOSE_SOURCE_URL = None
COMPOSE_MATTER = "cold_beta_equilibrated"
COMPOSE_INCLUDES_LEPTONS = False
COMPOSE_BARYON_DENSITY_MIN_FM3 = None
COMPOSE_BARYON_DENSITY_MAX_FM3 = None
COMPOSE_NATIVE_POINTS = 2001
COMPOSE_ORDERING_POLICY = "strict"

# Stellar experiment controls
VALIDATION_MODE = "strict"
ENABLE_BACKGROUND_DIAGNOSTIC = False
STAR_CENTRAL_PRESSURE_MEV_FM3 = 100.0
RETAIN_STAR_PROFILE = True
SEQUENCE_PRESSURE_RANGE_MEV_FM3 = None  # None uses the declared EoS domain.
SEQUENCE_POINTS = 9

# Advanced numerical controls. The 50 km ceiling is an explicit notebook
# override that lets all nine bundled-CSV demonstration points reach the
# positive-pressure source boundary; it is not a change to the EoS.
RADIUS_START_KM = 1.0e-4
RADIUS_MAX_KM = 50.0
CENTER_EXPANSION_LIMIT_KM = 1.0e-4
ODE_RTOL = 1.0e-10
ODE_ATOL = 1.0e-12
PROFILE_POINTS = 300

# Presentation and opt-in output
PLOT_SAMPLE_POINTS = 513
SAVE_RESULTS = False
OUTPUT_DIRECTORY = "notebook-output"
FIGURE_FORMATS = ("png", "svg")

### What may be changed

- **Analytical input:** change the full `P(epsilon)` expression, its consistent `dP/d(epsilon)`, model name, and finite energy-density domain in `analytical_eos.py`. The starter quadratic is only an example, not a required parameterization.
- **CSV input:** change the path and explicit column names. Energy density must mean total energy density including rest mass; energy density and pressure use MeV/fm^3.
- **One star:** change the central pressure, which must be above the source's minimum pressure and no greater than its maximum.
- **Sequence:** leave the range as `None` to sample the declared domain or provide a strictly increasing `(minimum, maximum)` pressure pair.
- **Numerics:** change the integration ceiling, tolerances, and retained profile size only with convergence checks.
- **CompOSE:** explicitly declare source identity, lepton inclusion, density selection, native sample count, and ordering policy. Missing fields remain missing.

Do not treat units, silent ordering changes, smoothing, clipping, extrapolation, crust splicing, or diagnostic validation as harmless plotting knobs.

In [ ]:
def find_repository_root(override=None):
    if override is not None:
        candidates = (Path(override).expanduser().resolve(),)
    else:
        start = Path.cwd().resolve()
        candidates = (start, *start.parents)
    for candidate in candidates:
        if (candidate / "pyproject.toml").is_file() and (
            candidate / "src" / "neutron_star_eos"
        ).is_dir():
            return candidate
    raise RuntimeError(
        "Could not locate the repository. Set REPOSITORY_ROOT explicitly."
    )


REPO_ROOT = find_repository_root(REPOSITORY_ROOT)


def resolve_user_path(value):
    candidate = Path(value).expanduser()
    return (candidate if candidate.is_absolute() else REPO_ROOT / candidate).resolve()


def display_path(path):
    try:
        return path.relative_to(REPO_ROOT).as_posix()
    except ValueError:
        return str(path)


if INPUT_KIND not in {"csv", "analytical", "compose"}:
    raise ValueError("INPUT_KIND must be 'csv', 'analytical', or 'compose'")
if VALIDATION_MODE not in {"strict", "background_diagnostic"}:
    raise ValueError("Unknown VALIDATION_MODE")
if isinstance(PLOT_SAMPLE_POINTS, bool) or PLOT_SAMPLE_POINTS < 17:
    raise ValueError("PLOT_SAMPLE_POINTS must be an integer of at least 17")

print("Repository:", REPO_ROOT)
print("Selected input kind:", INPUT_KIND)

## Load the selected EoS

For analytical work, edit `analytical_eos.py` rather than a package file. The loader below executes that user-authored Python file afresh, so no kernel restart or import-cache workaround is needed. It records both its exact byte hash and a cross-platform UTF-8/LF-normalized hash. Executing the file has the same trust implications as running any local Python script.

In [ ]:
def definition_hashes(path):
    raw = path.read_bytes()
    text = raw.decode("utf-8")
    canonical = text.replace("\r\n", "\n").replace("\r", "\n")
    return {
        "hash_policy": "utf8_text_normalized_lf_v1",
        "canonical_sha256": hashlib.sha256(canonical.encode("utf-8")).hexdigest(),
        "raw_sha256": hashlib.sha256(raw).hexdigest(),
    }


def load_analytical_model(path):
    before = definition_hashes(path)
    namespace = runpy.run_path(str(path), run_name="neutron_star_eos_user_definition")
    after = definition_hashes(path)
    if before != after:
        raise RuntimeError("The analytical definition changed while it was loading")
    build_model = namespace.get("build_model")
    if not callable(build_model):
        raise TypeError("analytical_eos.py must define callable build_model(...)")
    relative_path = display_path(path)
    description = str(
        namespace.get("SOURCE_DESCRIPTION", "User-defined analytical EoS")
    )
    source_identity = (
        f"{description}; definition={relative_path}; "
        f"normalized_lf_sha256={before['canonical_sha256']}"
    )
    loaded = build_model(source_identity=source_identity)
    if not isinstance(loaded, EosModel):
        raise TypeError("build_model(...) must return an EosModel")
    return loaded, {"repository_relative_path": relative_path, **before}

In [ ]:
definition_record = None
source_path = None

if INPUT_KIND == "csv":
    source_path = resolve_user_path(CSV_PATH)
    model = open_eos(
        source_path,
        kind="csv",
        name=CSV_NAME,
        source_description=CSV_SOURCE_DESCRIPTION,
        epsilon_column=CSV_EPSILON_COLUMN,
        pressure_column=CSV_PRESSURE_COLUMN,
        baryon_density_column=CSV_BARYON_DENSITY_COLUMN,
    )
elif INPUT_KIND == "analytical":
    source_path = resolve_user_path(ANALYTICAL_DEFINITION_PATH)
    model, definition_record = load_analytical_model(source_path)
else:
    if COMPOSE_PATH is None or COMPOSE_MODEL_ID is None or COMPOSE_SOURCE_URL is None:
        raise ValueError(
            "CompOSE input requires COMPOSE_PATH, COMPOSE_MODEL_ID, and "
            "COMPOSE_SOURCE_URL"
        )
    source_path = resolve_user_path(COMPOSE_PATH)
    model = open_eos(
        source_path,
        kind="compose",
        model_id=COMPOSE_MODEL_ID,
        source_url=COMPOSE_SOURCE_URL,
        matter=COMPOSE_MATTER,
        includes_leptons=COMPOSE_INCLUDES_LEPTONS,
        baryon_density_min_fm3=COMPOSE_BARYON_DENSITY_MIN_FM3,
        baryon_density_max_fm3=COMPOSE_BARYON_DENSITY_MAX_FM3,
        native_points=COMPOSE_NATIVE_POINTS,
        ordering_policy=COMPOSE_ORDERING_POLICY,
    )

print("Loaded source:", display_path(source_path))
if definition_record is not None:
    print("Analytical definition hash:", definition_record["canonical_sha256"])

## Identity, capabilities, and diagnostics

`available_with_diagnostics` preserves findings rather than hiding them. `unavailable` means the operation is not authorized by the available evidence; it does not erase thermodynamics that remain inspectable. Every later section checks the relevant capability before acting.

In [ ]:
report = model.report()
print(model.summary())
print("\nCapability details:")
for capability in report.capabilities:
    reason = f" -- {capability.reason}" if capability.reason else ""
    diagnostics = (
        f" [{', '.join(capability.diagnostic_codes)}]"
        if capability.diagnostic_codes
        else ""
    )
    print(f"  {capability.name:22s} {capability.status}{diagnostics}{reason}")
display(JSON(report.to_dict(), expanded=False))

## Thermodynamic inspection

Markers are authoritative source nodes when the input supplies them. Curves are deterministic evaluations or native-field reconstructions and remain separately labeled. Reference lines at `c_s^2 = 0` and the causal limit `c_s^2 = 1` are diagnostic guides, not data transformations.

In [ ]:
thermodynamic_view = None
thermodynamics_capability = report.capability("thermodynamics")
if thermodynamics_capability.available:
    thermodynamic_view = model.thermodynamics(curve_points=PLOT_SAMPLE_POINTS)
    for series in thermodynamic_view.series:
        print(
            f"{series.role}: {series.rows} rows; "
            f"columns={', '.join(series.column_names)}"
        )
        if series.diagnostic_codes:
            print("  diagnostics:", ", ".join(series.diagnostic_codes))
else:
    display(Markdown(f"**Thermodynamics skipped:** {thermodynamics_capability.reason}"))

In [ ]:
figures = {}
if thermodynamic_view is not None:
    has_continuous_curve = "continuous_barotrope" in thermodynamic_view.roles
    figure, axes = plt.subplots(1, 2, figsize=(12.0, 4.8), constrained_layout=True)
    plot_pressure_energy(
        model,
        ax=axes[0],
        curve_points=PLOT_SAMPLE_POINTS,
        show_source_nodes=True,
        show_stellar_barotrope=has_continuous_curve,
    )
    plot_sound_speed_squared(
        model,
        ax=axes[1],
        curve_points=PLOT_SAMPLE_POINTS,
        include_stellar_barotrope=has_continuous_curve,
    )
    figure.suptitle(model.model_name)
    figures["thermodynamics"] = figure
    plt.show()

### CompOSE-only views

Closure and cold-path residuals remain separate from the primary thermodynamic curves. Composition fields are plotted only when their capability is available, preserve NaNs, and are not all assumed to be fractions. Phase codes are categorical source information.

In [ ]:
if model.kind == "compose" and thermodynamic_view is not None:
    try:
        figure, axes = plt.subplots(1, 2, figsize=(12.0, 4.8), constrained_layout=True)
        plot_compose_closure_residuals(model, ax=axes[0])
        plot_compose_cold_residuals(model, ax=axes[1])
        figures["compose_residuals"] = figure
        plt.show()
    except (EosInputError, KeyError, ValueError) as error:
        display(Markdown(f"**CompOSE residual view unavailable:** {error}"))

    composition_capability = report.capability("composition")
    if composition_capability.available:
        figure, axis = plt.subplots(figsize=(7.0, 4.8), constrained_layout=True)
        plot_composition(model, ax=axis)
        figures["composition"] = figure
        plt.show()
    else:
        display(Markdown(f"**Composition skipped:** {composition_capability.reason}"))

    try:
        figure, axis = plt.subplots(figsize=(7.0, 3.8), constrained_layout=True)
        plot_phase_codes(model, ax=axis)
        figures["phase_codes"] = figure
        plt.show()
    except (EosInputError, KeyError, ValueError) as error:
        plt.close(figure)
        display(Markdown(f"**Phase-code view unavailable:** {error}"))

## One continuous stellar background

The exact numerical configuration is displayed before integration. Strict mode follows the `stellar_background` capability. A failing physics gate can be bypassed only for the toolkit's explicitly supported background diagnostic mode and only when both diagnostic controls were deliberately enabled; such a result retains its failing validation status.

In [ ]:
stellar_config = StellarConfig(
    radius_start_km=RADIUS_START_KM,
    radius_max_km=RADIUS_MAX_KM,
    center_expansion_limit_km=CENTER_EXPANSION_LIMIT_KM,
    ode_rtol=ODE_RTOL,
    ode_atol=ODE_ATOL,
    profile_points=PROFILE_POINTS,
)
display(JSON(asdict(stellar_config), expanded=True))

stellar_capability = report.capability("stellar_background")
diagnostic_override = (
    not stellar_capability.available
    and VALIDATION_MODE == "background_diagnostic"
    and ENABLE_BACKGROUND_DIAGNOSTIC
    and model.barotrope is not None
)
stellar_authorized = stellar_capability.available or diagnostic_override
if diagnostic_override:
    display(
        Markdown(
            "**Diagnostic run enabled:** this does not upgrade the EoS physics "
            "validation and must not be presented as a validated stellar result."
        )
    )

In [ ]:
star = None
if stellar_authorized:
    try:
        star = model.solve_star(
            central_pressure_mev_fm3=STAR_CENTRAL_PRESSURE_MEV_FM3,
            config=stellar_config,
            retain_profile=RETAIN_STAR_PROFILE,
            validation_mode=VALIDATION_MODE,
        )
    except (EosInputError, StellarSolveError, ArithmeticError) as error:
        display(Markdown(f"**Star unavailable:** {error}"))
    else:
        print(f"Mass: {star.mass_msun:.8g} Msun")
        print(f"Source-boundary radius: {star.radius_km:.8g} km")
        print("Boundary status:", star.boundary_status)
        print("EoS validation status:", star.eos_validation_status)
else:
    display(Markdown(f"**Stellar calculation skipped:** {stellar_capability.reason}"))

if star is not None and star.radius_profile_km:
    figure, axis = plt.subplots(figsize=(7.0, 4.8), constrained_layout=True)
    plot_mass_profile(star, ax=axis)
    figures["mass_profile"] = figure
    plt.show()

## Central-pressure sequence

This is a requested collection of background integrations, not a stability analysis. Every requested pressure remains in the result. Failed attempts are shown with their reasons, curves never bridge missing attempts, and no point is labeled a maximum mass.

In [ ]:
sequence = None
if stellar_authorized:
    try:
        if SEQUENCE_PRESSURE_RANGE_MEV_FM3 is None:
            requested_pressures = None
        else:
            lower_pressure, upper_pressure = map(float, SEQUENCE_PRESSURE_RANGE_MEV_FM3)
            if not 0.0 < lower_pressure < upper_pressure:
                raise ValueError(
                    "Sequence pressure range must satisfy 0 < lower < upper"
                )
            requested_pressures = np.geomspace(
                lower_pressure, upper_pressure, int(SEQUENCE_POINTS)
            )
        sequence = model.solve_sequence(
            requested_pressures,
            points=SEQUENCE_POINTS,
            config=stellar_config,
            validation_mode=VALIDATION_MODE,
        )
    except (EosInputError, RuntimeError, ArithmeticError, ValueError) as error:
        display(Markdown(f"**Sequence unavailable:** {error}"))

if sequence is not None:
    print("Sequence status:", sequence.status)
    print("pressure [MeV/fm^3] | status      | mass [Msun] | radius [km] | reason")
    for attempt in sequence.attempts:
        if attempt.star is None:
            mass_text, radius_text = "-", "-"
        else:
            mass_text = f"{attempt.star.mass_msun:.8g}"
            radius_text = f"{attempt.star.radius_km:.8g}"
        print(
            f"{attempt.central_pressure_mev_fm3:20.8g} | "
            f"{attempt.status:11s} | {mass_text:11s} | {radius_text:11s} | "
            f"{attempt.reason or ''}"
        )

In [ ]:
if sequence is not None:
    figure, axes = plt.subplots(1, 2, figsize=(12.0, 4.8), constrained_layout=True)
    plot_mass_radius(
        sequence,
        ax=axes[0],
        connect=False,
        color_by_central_pressure=True,
    )
    plot_sequence_status(sequence, ax=axes[1])
    figure.suptitle(f"{model.model_name}: source-boundary backgrounds")
    figures["sequence"] = figure
    plt.show()

## Optional reproducible output

Set `SAVE_RESULTS = True` only after choosing a new destination. The toolkit refuses to overwrite an existing result directory. Analytical runs copy the authoritative definition file and record both file hashes alongside the toolkit's evaluated-callable fingerprint.

In [ ]:
saved_directory = None
if SAVE_RESULTS:
    destination = resolve_user_path(OUTPUT_DIRECTORY)
    if sequence is not None:
        saved_directory = model.write_sequence(destination, sequence)
        saved_result_kind = "sequence"
    elif star is not None:
        saved_directory = model.write_star(destination, star)
        saved_result_kind = "star"
    else:
        saved_directory = model.write_inspection(destination)
        saved_result_kind = "inspection"

    figure_directory = saved_directory / "figures"
    figure_directory.mkdir()
    for figure_name, figure in figures.items():
        for suffix in FIGURE_FORMATS:
            normalized_suffix = str(suffix).lower().lstrip(".")
            if normalized_suffix not in {"png", "svg", "pdf"}:
                raise ValueError(f"Unsupported figure format: {suffix}")
            save_options = {"dpi": 300} if normalized_suffix == "png" else {}
            figure.savefig(
                figure_directory / f"{figure_name}.{normalized_suffix}",
                **save_options,
            )

    copied_definition = None
    if definition_record is not None:
        copied_path = saved_directory / "analytical_eos.py"
        shutil.copy2(source_path, copied_path)
        copied_definition = {
            "copied_filename": copied_path.name,
            **definition_hashes(copied_path),
        }
        if (
            copied_definition["canonical_sha256"]
            != definition_record["canonical_sha256"]
        ):
            raise RuntimeError("Copied analytical definition failed hash verification")

    manifest = {
        "schema_version": "eos-notebook-experiment-v1",
        "input_kind": model.kind,
        "result_kind": saved_result_kind,
        "source_path": display_path(source_path),
        "analytical_definition": definition_record,
        "copied_analytical_definition": copied_definition,
        "parameters": {
            "validation_mode": VALIDATION_MODE,
            "star_central_pressure_mev_fm3": STAR_CENTRAL_PRESSURE_MEV_FM3,
            "sequence_pressure_range_mev_fm3": SEQUENCE_PRESSURE_RANGE_MEV_FM3,
            "sequence_points": SEQUENCE_POINTS,
            "stellar_config": asdict(stellar_config),
            "plot_sample_points": PLOT_SAMPLE_POINTS,
        },
        "model_report": report.to_dict(),
    }
    (saved_directory / "experiment.json").write_text(
        json.dumps(manifest, indent=2, sort_keys=True, allow_nan=False) + "\n",
        encoding="utf-8",
        newline="\n",
    )
    print("Saved experiment:", saved_directory)
else:
    print("SAVE_RESULTS is False; no files were written.")

In [ ]:
default_csv_run = (
    INPUT_KIND == "csv"
    and source_path == (REPO_ROOT / "examples" / "tabulated.csv").resolve()
    and STAR_CENTRAL_PRESSURE_MEV_FM3 == 100.0
    and SEQUENCE_PRESSURE_RANGE_MEV_FM3 is None
    and SEQUENCE_POINTS == 9
    and RADIUS_MAX_KM == 50.0
)
if default_csv_run:
    assert thermodynamic_view is not None
    assert star is not None
    assert sequence is not None and sequence.status == "complete"
    assert len(sequence.attempts) == SEQUENCE_POINTS

print("EOS_NOTEBOOK_EXECUTION_OK")